# ToolUse Agent using Anthropic models

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction TB

    INIT --> HAS_MESSAGE
    HAS_MESSAGE --> CHAT: condition_true
    HAS_MESSAGE --> TOOL_USE: condition_false

    CHAT--> IS_TOOL_CALL
    
    IS_TOOL_CALL --> FINAL: condition_false
    IS_TOOL_CALL --> TOOL_USE: condition_true

    TOOL_USE --> FINAL


```

### State Diagram (User View)
```mermaid
stateDiagram-v2
direction TB
    state "Start Agent" as StartAgent
    state "Continue Agent" as ContinueAgent
    state "Interrupt Agent" as InterruptAgent

    Initialize --> StartAgent : start_async
    StartAgent --> ContinueAgent : continue_async
    ContinueAgent --> InterruptAgent : interrupt_async
    InterruptAgent --> ContinueAgent : continue_async
    ContinueAgent --> ContinueAgent : continue_async
    ContinueAgent --> Terminate
    Terminate --> StartAgent : start_async
    Terminate --> InterruptAgent : interrupt_async

```



### a) Create Agent

In [1]:
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
mcp_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await mcp_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    project_name="gai",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    mcp_client=mcp_client,
)

### reset monologue (optional)

In [2]:
ToolUseAgent.reset(project_name="gai",agent_name="ToolUseAgent")


### b) start_async

In [ ]:
goal = "When is the next public holiday in Singapore?"
user_message=f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
"""

resp=await agent.start_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)

I'll help you find the next public holiday in Singapore. Let me first get the current date and then search for Singapore's public holidays.Now let me search for Singapore's public holidays to find the next one after July 1, 2025.

### c) Show monologue

In [5]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "4017360d-7420-4751-9261-e60a91b2ad10",
    "header": {
        "sender": "User",
        "recipient": "Assistant",
        "timestamp": 1751305976.6207695,
        "order": 0
    },
    "body": {
        "type": "state",
        "state_name": "TOOL_USE",
        "step_no": 9,
        "content_type": "text",
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": "toolu_01Y38SGjdpFX5a29oxsULzH6",
                "content": "Use SGT"
            }
        ]
    }
}
{
    "id": "fa0b4555-7fd6-420b-b51e-a6cbde34ce7e",
    "header": {
        "sender": "Assistant",
        "recipient": "User",
        "timestamp": 1751305980.772542,
        "order": 0
    },
    "body": {
        "type": "state",
        "state_name": "TOOL_USE",
        "step_no": 9,
        "content_type": "text",
        "role": "assistant",
        "content": [
      

### d) continue_async

In [4]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


Based on the search results, I can see the complete list of Singapore's public holidays for 2025. Since today is July 1, 2025, let me identify the next public holiday.

Looking at the 2025 public holidays:
- New Year's Day: January 1, 2025 (already passed)
- Chinese New Year: January 29-30, 2025 (already passed)
- Hari Raya Puasa: March 31, 2025 (already passed)
- Good Friday: April 18, 2025 (already passed)
- Labour Day: May 1, 2025 (already passed)
- Polling Day: May 3, 2025 (already passed)
- Vesak Day: May 12, 2025 (already passed)
- Hari Raya Haji: June 7, 2025 (already passed)

The next public holiday after July 1, 2025 is:

**National Day - Saturday, August 9, 2025**

This is Singapore's National Day, celebrating the country's independence. It falls on a Saturday this year.

### e) interrupt_async

Should be able to interrupt the agent and continue with original task.

In [6]:
resp = await agent.interrupt_async(user_message="Tell me a one paragraph joke.")
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


ERROR    async_anthropic_create: Error while calling anthropic API: Error code: 400 - {'type': 'error', 'error':   
         {'type': 'invalid_request_error', 'message': 'messages.6: `tool_use` ids were found without `tool_result` 
         blocks immediately after: toolu_01LXKAKvDarw5nvCMsAaZapS. Each `tool_use` block must have a corresponding 
         `tool_result` block in the next message.'}}

WARNING  [tool_use] async_anthropic_create: Error while calling anthropic API: Error code: 400 - {'type': 'error', 
         'error': {'type': 'invalid_request_error', 'message': 'messages.6: `tool_use` ids were found without      
         `tool_result` blocks immediately after: toolu_01LXKAKvDarw5nvCMsAaZapS. Each `tool_use` block must have a 
         corresponding `tool_result` block in the next message.'}}. Retry in 5 seconds (1/5).

Here's a one paragraph joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, but sees nothing except regular bookshelves. He turns back to the librarian confused, and she points to the shelf directly behind where he was standing and says, "No, literally - the paranoia books are right behind you on that shelf. This is the psychology section."

Now, let me get back on track and answer your original question about Singapore's next public holiday.

Based on the search results from the Ministry of Manpower (MOM), since today is July 1, 2025, the **next public holiday in Singapore is National Day on Saturday, August 9, 2025**.

Here's the complete list of remaining public holidays for 2025 after July 1st:
- **National Day: August 9, 2025 (Saturday)**
- Deepavali: October 20, 2025 (Monday)
- Christmas Day: December 25, 2025 (Thursday)

So National Day is coming up in about 5 weeks from no

---
## Scenario 1

- LLM Interrupts itself to ask user question
- User cannot continue because LLM is waiting for user input

In [1]:
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
mcp_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await mcp_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    project_name="gai",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    mcp_client=mcp_client,
)

In [2]:
ToolUseAgent.reset(project_name="gai", agent_name="ToolUseAgent")

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = await agent.start_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


I'll help you get the current time. First, let me ask you which timezone you'd like to see the time for.What timezone would you like to see the current time for? Please provide the timezone (for example: America/New_York, Europe/London, Asia/Tokyo, UTC, etc.)

Confirm that last message is "user_input".

In [3]:
message = agent.fsm.monologue.list_messages()[-1]
found = next(
    (content for content in message.body.content if content.get("name") == "user_input"),
    None,
)
print(found)
assert found

{'id': 'toolu_016Udhpsro8Eb1ghYmkfawF9', 'input': {}, 'name': 'user_input', 'type': 'tool_use'}


Confirm that "continue_async" will not work.

In [4]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


---
## Scenario 2

- LLM Interrupts itself to ask user question
- User responds with answer

In [1]:
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
mcp_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await mcp_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    project_name="gai",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    mcp_client=mcp_client,
)

In [2]:
ToolUseAgent.reset(project_name="gai", agent_name="ToolUseAgent")

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = await agent.start_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


I'll help you get the current time. First, let me ask for your timezone preference.What timezone would you like me to use for the current time? Please provide an IANA timezone name (for example: America/New_York, Europe/London, Asia/Tokyo, America/Los_Angeles, etc.), or if you're not sure, just tell me your location or preferred timezone.

Confirm that last message is "user_input".

In [3]:
message = agent.fsm.monologue.list_messages()[-1]
found = next(
    (content for content in message.body.content if content.get("name") == "user_input"),
    None,
)
print(found)
assert found

{'id': 'toolu_01Y38SGjdpFX5a29oxsULzH6', 'input': {}, 'name': 'user_input', 'type': 'tool_use'}


Confirm that "continue_async" with message will work.

In [4]:
resp = await agent.continue_async(user_message="Use SGT")
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


Perfect! I'll get the current time in Singapore Time (SGT) for you.

Continue to get the time.

In [6]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


The current time in Singapore Time (SGT) is:

**July 1, 2025 at 1:54:18 AM**

Confirm if its end of response by continuing the agent.

In [7]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
